In [80]:
TRAINING_FINISHED = False
RANDOM_STATE = 42
MAX_ITERATIONS = 5000
MAX_ITERATIONS_SVM = 1000000
TRAINING_SET_SIZE = 0.7
VALIDATION_TEST_SET_RATIO = 0.5

In [81]:
import pandas as pd

data = pd.read_csv('reviews.csv', sep=",",
                   usecols=['rating', 'review_text', 'helpful', 'review_date'])


In [66]:
data['rating'] = data['rating'].fillna(0)
data['helpful'] = data['helpful'].fillna(0)
data['review_text'] = data['review_text'].fillna("")

In [82]:
# lowercase conversion removed: casing can carry sentiment signal (e.g., "GREAT" vs "great")

In [83]:
import contractions

# expand contractions
data['review_text'] = data['review_text'].apply(contractions.fix)

In [84]:
import emoji

# interpret emojis as words
data['review_text'] = data['review_text'].apply(emoji.demojize)

In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

y = data['rating']

In [86]:
from sklearn.model_selection import train_test_split, cross_val_score

# Split raw data first (before vectorization to prevent data leakage)
text_train, text_temp, helpful_train, helpful_temp, y_train, y_temp = train_test_split(
    data['review_text'], data['helpful'], y,
    test_size=1 - TRAINING_SET_SIZE, random_state=RANDOM_STATE, stratify=y
)

text_val, text_test, helpful_val, helpful_test, y_validation, y_test = train_test_split(
    text_temp, helpful_temp, y_temp,
    test_size=VALIDATION_TEST_SET_RATIO, random_state=RANDOM_STATE, stratify=y_temp
)

# Fit vectorizer on training data only, then transform all sets
X_text_train = tfidf_vectorizer.fit_transform(text_train)
X_text_val = tfidf_vectorizer.transform(text_val)
X_text_test = tfidf_vectorizer.transform(text_test)

# Combine text features with helpful column
X_train = hstack([X_text_train, csr_matrix(helpful_train.values.reshape(-1, 1))])
X_validation = hstack([X_text_val, csr_matrix(helpful_val.values.reshape(-1, 1))])
X_test = hstack([X_text_test, csr_matrix(helpful_test.values.reshape(-1, 1))])

In [87]:
models = []

In [88]:
from sklearn.svm import SVC

# Support Vector Machine model
svc_model = SVC(
    kernel='linear',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=MAX_ITERATIONS_SVM
)
svc_model.fit(X_train, y_train)
models.append(svc_model)

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [89]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
logreg_model = LogisticRegression(class_weight='balanced', max_iter=MAX_ITERATIONS)
logreg_model.fit(X_train, y_train)
models.append(logreg_model)

In [90]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    max_depth=9,
    class_weight='balanced',
    n_estimators=300
)
rf_model.fit(X_train, y_train)
models.append(rf_model)

In [91]:
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd
from scipy import sparse

y_vals = y_train.values if hasattr(y_train, 'values') else np.array(y_train)
class_counts = pd.Series(y_vals).value_counts()
max_count = class_counts.max()

balanced_X, balanced_y = [], []
for cls in class_counts.index:
    mask = y_vals == cls
    X_cls = X_train[mask]
    y_cls = y_vals[mask]
    if len(y_cls) < max_count:
        X_cls, y_cls = resample(X_cls, y_cls, replace=True, n_samples=max_count, random_state=RANDOM_STATE)
    balanced_X.append(X_cls)
    balanced_y.append(y_cls)

if sparse.issparse(X_train):
    X_train_bal = sparse.vstack(balanced_X)
else:
    X_train_bal = np.vstack(balanced_X)
y_train_bal = np.concatenate(balanced_y)

architectures = [
    (64,),
    (128, 64),
    (256, 128),
    (128, 64, 32),
    (128, 128, 64),
    (128, 128, 32),
]
best_score = 0
best_mlp = None

for i, arch in enumerate(architectures):
    print(f"Training architecture #{i} {arch}:")
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=MAX_ITERATIONS,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    mlp.fit(X_train_bal, y_train_bal)

    val_score = accuracy_score(y_validation, mlp.predict(X_validation))
    print(f"- Validation Accuracy: {val_score:.4f}")

    if val_score > best_score:
        best_score = val_score
        best_mlp = mlp

models.append(best_mlp)

Training architecture #0 (64,):
- Validation Accuracy: 0.4828
Training architecture #1 (128, 64):
- Validation Accuracy: 0.5011
Training architecture #2 (256, 128):
- Validation Accuracy: 0.4861
Training architecture #3 (128, 64, 32):
- Validation Accuracy: 0.4968
Training architecture #4 (128, 128, 64):
- Validation Accuracy: 0.4871
Training architecture #5 (128, 128, 32):
- Validation Accuracy: 0.4828


In [92]:
from sklearn.metrics import accuracy_score, classification_report

# PLOT THAT instead
for i, model in enumerate(models):
    print(f'Evaluate Model #{i}: {type(model).__name__}')
    # SVC uses scaled data
    y_pred = model.predict(X_validation)
    print(f'Accuracy: {accuracy_score(y_validation, y_pred):.4f}')
    print(classification_report(y_validation, y_pred))

    # print("Cross-validation scores: {}".format(cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')))
    # print("Confusion matrix:\n{}".format(confusion_matrix(y_validation, y_pred)))


Evaluate Model #0: SVC
Accuracy: 0.5086
              precision    recall  f1-score   support

           1       0.53      0.64      0.58       237
           2       0.10      0.10      0.10        70
           3       0.17      0.29      0.21        76
           4       0.14      0.14      0.14        95
           5       0.80      0.62      0.70       454

    accuracy                           0.51       932
   macro avg       0.35      0.36      0.35       932
weighted avg       0.56      0.51      0.53       932

Evaluate Model #1: LogisticRegression
Accuracy: 0.4989
              precision    recall  f1-score   support

           1       0.55      0.62      0.58       237
           2       0.15      0.20      0.17        70
           3       0.18      0.36      0.24        76
           4       0.11      0.11      0.11        95
           5       0.81      0.59      0.68       454

    accuracy                           0.50       932
   macro avg       0.36      0.37   

In [93]:
# FINAL SCORING; ONLY USE AFTER FINISHED TRAINING
if not TRAINING_FINISHED:
    print("Skipping final test — set TRAINING_FINISHED = True when ready.")
else:
    for i, model in enumerate(models):
        print(f'Evaluate Model #{i} with type: {type(model).__name__}')
        y_pred = model.predict(X_test)
        print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
        print(classification_report(y_test, y_pred))

Skipping final test — set TRAINING_FINISHED = True when ready.
